# Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'


## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

# Parameters

In [2]:
start_date = '20240322'
end_date = '20240510'

local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/'

In [3]:
category_list = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_v1/category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list['categories_l1'] = category_list['categories_l1'].str.lower()
category_list['categories_l2'] = category_list['categories_l2'].str.lower()
category_list = category_list[['app_name', 'categories_l1', 'categories_l2']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(433, 3)
['educational' 'ott' 'delivery_grocery' 'tools' 'office' 'dating' 'news'
 'devotional' 'travel_metro' 'gaming' 'travel_bookings'
 'finance_transactions' 'ecommerce' 'streaming_music' 'finance_investment'
 'health' 'fantasy_sports' 'delivery_food' 'home_security' 'finance_news'
 'travel_ridehailing' 'social' 'telecom' 'entertainment' 'news_finance'
 'wearable' 'driver_app' 'service']
['educational' 'rest' 'office' 'ride haling' 'gaming' 'finance_investment'
 'driver_app']


,app_name,categories_l1,categories_l2
0,ignou e-content,educational,educational


# Data

### Scratch Card

In [4]:
scratchcards = f"""
    
    WITH unlocked AS (

    SELECT
        user_id
        -- rewards_transaction_id order_id
    FROM 
        raw.kafka_pricing_scratchcards_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND rewards_state = 'UNLOCKED'
        AND rewards_type = 'EXTERNAL_REWARD'
        AND JSON_EXTRACT_SCALAR(rewards, '$.externalRewardMetaData.externalRewardId') IN (  '6603f871448cb38c86d1f0d0', '65fc8e0a5d26549b39e572de', '660c0c93639d07858bc1d721', '660c0c1c7c0420ce6f6c7a5d', '6603faa7639d07858bbd37a9',
                                                                                            '660c0d056d7ddb4719d4eab3', '660c0e5c712d0067117c2de5', '65fc8536c3c58734495647d7', '6603f9af712d00671177355d', '65fc8d19fb638769daf4bb1e',
                                                                                            '66311e60b397b66e22161cd5', '65fc8a50d53292624c41d395', '6603fd28639d07858bbea5c0', '65fac1cce4838dd72578e34c', '65e85625a1bed81a09450b32',
                                                                                            '65d457d4d79b4ea79b5a6b09', '65cb6dc116c7594d528d8d50', '65e85b3a78839f742cac6cb4', '65d31c5fc526f1e342f2d9cb', '65e847c39bb6821c785b9e21')
    GROUP BY 1
    ),
    
    
    scratched AS (
    
    SELECT
        user_id
        -- rewards_transaction_id order_id

    FROM 
        raw.kafka_pricing_scratchcards_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND rewards_type = 'EXTERNAL_REWARD'
        AND rewards_state = 'SCRATCHED' 
        AND JSON_EXTRACT_SCALAR(rewards, '$.externalRewardMetaData.externalRewardId') IN (  '6603f871448cb38c86d1f0d0', '65fc8e0a5d26549b39e572de', '660c0c93639d07858bc1d721', '660c0c1c7c0420ce6f6c7a5d', '6603faa7639d07858bbd37a9',
                                                                                            '660c0d056d7ddb4719d4eab3', '660c0e5c712d0067117c2de5', '65fc8536c3c58734495647d7', '6603f9af712d00671177355d', '65fc8d19fb638769daf4bb1e',
                                                                                            '66311e60b397b66e22161cd5', '65fc8a50d53292624c41d395', '6603fd28639d07858bbea5c0', '65fac1cce4838dd72578e34c', '65e85625a1bed81a09450b32',
                                                                                            '65d457d4d79b4ea79b5a6b09', '65cb6dc116c7594d528d8d50', '65e85b3a78839f742cac6cb4', '65d31c5fc526f1e342f2d9cb', '65e847c39bb6821c785b9e21')
    GROUP BY 1
    ),
    
    rewards AS (
    
    SELECT
        event_props_user_id user_id,
        MAX(CASE WHEN event_props_action = 'ctaClicked' THEN event_props_user_id END) AS cta_clicked_user_id,
        MAX(CASE WHEN event_props_action = 'copyCodeClicked' THEN event_props_user_id END) AS copy_code_clicked_user_id
        
    FROM 
        clevertap.customer_rewards_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND event_props_action IN ('ctaClicked', 'copyCodeClicked')
        AND event_props_reward_partner in ('My11 Circle', 'My 11Circle')
        
    GROUP BY 1
    )
    
    SELECT  
        unlocked.user_id unlocked_customer_id,
        COALESCE(scratched.user_id, rewards.user_id) scratched_customer_id,
        rewards.user_id rewards_customer_id,
        cta_clicked_user_id,
        copy_code_clicked_user_id
    FROM 
        unlocked 
    LEFT JOIN scratched
        ON unlocked.user_id = scratched.user_id
    LEFT JOIN rewards
        ON unlocked.user_id = rewards.user_id
"""
df_scratchcards = pd.read_sql(scratchcards, connection)

In [5]:
df_scratchcards.head()

,unlocked_customer_id,scratched_customer_id,rewards_customer_id,cta_clicked_user_id,copy_code_clicked_user_id
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,None,None,None
1,5cb02eb654bc7263ff34a5ff,None,None,None,None
2,65e73c6bf5efdbc08f2f1317,None,None,None,None
3,641bc23ed8f1b3f1dee45fc2,None,None,None,None
4,5d576ca6bde10e0d544289f7,5d576ca6bde10e0d544289f7,None,None,None


In [41]:
# df_scratchcards.to_csv('df_scratchcards.csv', index=False)
df_scratchcards = pd.read_csv('df_scratchcards.csv')

### Appo

In [43]:
local_database = '/Users/rapido/local-datasets/customer/appography/appography_v2/bangalore/database/'
# df_customer_appography_data.to_csv(local_database + 'raw_customer_appography_data.csv', index=False)
df_customer_appography_data = pd.read_csv(local_database + 'raw_customer_appography_data.csv')

# regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customer_appography_data['app_list_set'] = df_customer_appography_data['app_list_set'].apply(lambda x: extract_and_trim(x))

In [45]:
df_customer_appography_data

,customer_id,app_list_set
0,5737c6adddbec2203f73316d,"[whatsapp, paytm money, samsung galaxy wearabl..."
1,5737c6aeddbec2203f733176,"[outlook, onedrive, ola, microsoft 365, groww,..."
2,5737c6b0ddbec2203f733182,"[wynk music, messenger, mx takatak, google cal..."
3,5737c6b1ddbec2203f73318b,"[snapchat, instagram, google photos, zoom, my1..."
4,5737c6baddbec2203f7331d9,"[google sheets, jio cinema, paytm money, googl..."
...,...,...
1878610,6609aa278fcbab0f6302fbe0,"[google photos, uber, flipkart, google maps, p..."
1878611,6609aa86a9381bb7123b1d14,"[whatsapp, instagram, messenger, snapchat, jio..."
1878612,6609aab4db0e9caafc405cd8,"[facebook, snapchat, google maps, samsung gala..."
1878613,6609aacc8fcbabbbfa031371,"[brainly, zomato, spotify, wynk music, google ..."


In [47]:
df1 = df_scratchcards[~df_scratchcards['scratched_customer_id'].isna()]

In [51]:
df1.reset_index(inplace=True)

In [53]:
df1.drop('index', axis=1, inplace=True)

In [54]:
df1

,unlocked_customer_id,scratched_customer_id,rewards_customer_id,cta_clicked_user_id,copy_code_clicked_user_id
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN
1,5d576ca6bde10e0d544289f7,5d576ca6bde10e0d544289f7,NaN,NaN,NaN
2,6505907a3134867e03caacf2,6505907a3134867e03caacf2,NaN,NaN,NaN
3,63483e6f0229b1fe4944c38b,63483e6f0229b1fe4944c38b,NaN,NaN,NaN
4,65f5c8480f9aeb073b0aaf16,65f5c8480f9aeb073b0aaf16,NaN,NaN,NaN
...,...,...,...,...,...
1056834,635bc4c385b399d2a6534ee6,635bc4c385b399d2a6534ee6,NaN,NaN,NaN
1056835,61eecc596916b8683faa53a8,61eecc596916b8683faa53a8,NaN,NaN,NaN
1056836,62d030de7bdc24d87f3c8c2e,62d030de7bdc24d87f3c8c2e,NaN,NaN,NaN
1056837,61a1debb17dd6a9f8e02c43e,61a1debb17dd6a9f8e02c43e,NaN,NaN,NaN


### DE

In [55]:
df_appo_data = pd.merge(df1,
                        df_customer_appography_data,
                        how = 'inner',
                        left_on = 'scratched_customer_id',
                        right_on = 'customer_id'
                       )
df_appo_data

,unlocked_customer_id,scratched_customer_id,rewards_customer_id,cta_clicked_user_id,copy_code_clicked_user_id,customer_id,app_list_set
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,"[google maps, spotify, namma yatri, bookmyshow..."
1,5d576ca6bde10e0d544289f7,5d576ca6bde10e0d544289f7,NaN,NaN,NaN,5d576ca6bde10e0d544289f7,"[porter driver app, google calendar, jio cinem..."
2,6505907a3134867e03caacf2,6505907a3134867e03caacf2,NaN,NaN,NaN,6505907a3134867e03caacf2,"[dropbox, groww, truecaller, grofers, notion, ..."
3,63483e6f0229b1fe4944c38b,63483e6f0229b1fe4944c38b,NaN,NaN,NaN,63483e6f0229b1fe4944c38b,"[jiomart, goibibo, youtube, google news, adobe..."
4,5dba2be934054849200ef9a5,5dba2be934054849200ef9a5,NaN,NaN,NaN,5dba2be934054849200ef9a5,"[facebook, ola, disney+ hotstar, wynk music, g..."
...,...,...,...,...,...,...,...
189754,65e0496f113cbe2194d4e3ae,65e0496f113cbe2194d4e3ae,NaN,NaN,NaN,65e0496f113cbe2194d4e3ae,"[microsoft teams, wynk music, snapchat, trueca..."
189755,66068bec12bd0f6364109426,66068bec12bd0f6364109426,NaN,NaN,NaN,66068bec12bd0f6364109426,"[instagram, google maps, amazon prime video, z..."
189756,634cc330be6aa75b532eb84f,634cc330be6aa75b532eb84f,NaN,NaN,NaN,634cc330be6aa75b532eb84f,"[zomato, gmail, snapchat, spotify, youtube, di..."
189757,630100e745985b54cbc40825,630100e745985b54cbc40825,NaN,NaN,NaN,630100e745985b54cbc40825,"[gmail, google maps, whatsapp, paytm money, cl..."


# User Defined Function()

In [56]:
def calculate_percentage(df, numerator_col, denominator_col):
    return (df[numerator_col] * 100.00 / df[denominator_col]).round(2)

# Analysis

In [57]:
df_scratchcards.head(2)

,unlocked_customer_id,scratched_customer_id,rewards_customer_id,cta_clicked_user_id,copy_code_clicked_user_id
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN
1,5cb02eb654bc7263ff34a5ff,NaN,NaN,NaN,NaN


In [58]:
df_scratchcards['date_range'] = start_date + ' to ' + end_date

In [59]:
df_overall_numbers =  df_scratchcards\
                            .groupby(['date_range'])\
                            .agg(unlocked_customers = ('unlocked_customer_id', 'nunique'),
                                 scratched_customers = ('scratched_customer_id', 'nunique'),
                                 rewards_customers = ('rewards_customer_id', 'nunique'),
                                 cta_customers = ('cta_clicked_user_id', 'nunique'),
                                 copyCodeClicked_customers = ('copy_code_clicked_user_id', 'nunique')
                                )\
                            .reset_index()

df_overall_numbers

,date_range,unlocked_customers,scratched_customers,rewards_customers,cta_customers,copyCodeClicked_customers
0,20240322 to 20240510,6508503,1056839,122598,78490,120726


In [60]:
df_overall_numbers['% scratched_customers'] = calculate_percentage(df_overall_numbers, 'scratched_customers', 'unlocked_customers')
df_overall_numbers['% rewards code/click'] = calculate_percentage(df_overall_numbers, 'rewards_customers', 'scratched_customers')
df_overall_numbers['% rewards cta'] = calculate_percentage(df_overall_numbers, 'cta_customers', 'scratched_customers')
df_overall_numbers['% rewards copyCodeClicked'] = calculate_percentage(df_overall_numbers, 'copyCodeClicked_customers', 'scratched_customers')

## Overall

In [61]:
df_overall_numbers[['date_range','unlocked_customers','scratched_customers',
                    'rewards_customers','cta_customers','copyCodeClicked_customers']]

,date_range,unlocked_customers,scratched_customers,rewards_customers,cta_customers,copyCodeClicked_customers
0,20240322 to 20240510,6508503,1056839,122598,78490,120726


In [62]:
df_overall_numbers[['date_range','% scratched_customers','% rewards code/click',
                    '% rewards cta','% rewards copyCodeClicked']]

,date_range,% scratched_customers,% rewards code/click,% rewards cta,% rewards copyCodeClicked
0,20240322 to 20240510,16.24,11.6,7.43,11.42


## copyCodeClicked Customer 

In [63]:
df_appo_data

,unlocked_customer_id,scratched_customer_id,rewards_customer_id,cta_clicked_user_id,copy_code_clicked_user_id,customer_id,app_list_set
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,"[google maps, spotify, namma yatri, bookmyshow..."
1,5d576ca6bde10e0d544289f7,5d576ca6bde10e0d544289f7,NaN,NaN,NaN,5d576ca6bde10e0d544289f7,"[porter driver app, google calendar, jio cinem..."
2,6505907a3134867e03caacf2,6505907a3134867e03caacf2,NaN,NaN,NaN,6505907a3134867e03caacf2,"[dropbox, groww, truecaller, grofers, notion, ..."
3,63483e6f0229b1fe4944c38b,63483e6f0229b1fe4944c38b,NaN,NaN,NaN,63483e6f0229b1fe4944c38b,"[jiomart, goibibo, youtube, google news, adobe..."
4,5dba2be934054849200ef9a5,5dba2be934054849200ef9a5,NaN,NaN,NaN,5dba2be934054849200ef9a5,"[facebook, ola, disney+ hotstar, wynk music, g..."
...,...,...,...,...,...,...,...
189754,65e0496f113cbe2194d4e3ae,65e0496f113cbe2194d4e3ae,NaN,NaN,NaN,65e0496f113cbe2194d4e3ae,"[microsoft teams, wynk music, snapchat, trueca..."
189755,66068bec12bd0f6364109426,66068bec12bd0f6364109426,NaN,NaN,NaN,66068bec12bd0f6364109426,"[instagram, google maps, amazon prime video, z..."
189756,634cc330be6aa75b532eb84f,634cc330be6aa75b532eb84f,NaN,NaN,NaN,634cc330be6aa75b532eb84f,"[zomato, gmail, snapchat, spotify, youtube, di..."
189757,630100e745985b54cbc40825,630100e745985b54cbc40825,NaN,NaN,NaN,630100e745985b54cbc40825,"[gmail, google maps, whatsapp, paytm money, cl..."


In [64]:
## Explode list into individual rows

df_appography_explode = df_appo_data.explode('app_list_set')
df_appography_explode['app_list_set'] = df_appography_explode['app_list_set'].str.lower()

df_appography_explode.head(4)

,unlocked_customer_id,scratched_customer_id,rewards_customer_id,cta_clicked_user_id,copy_code_clicked_user_id,customer_id,app_list_set
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,google maps
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,spotify
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,namma yatri
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,bookmyshow


In [65]:
category_list

,app_name,categories_l1,categories_l2
0,ignou e-content,educational,educational
1,aha,ott,rest
2,grofers delivery app,delivery_grocery,rest
3,qr code reader,tools,rest
4,nishtha,office,office
5,brainly,educational,educational
6,hinge,dating,rest
7,wunderlist,office,office
8,slack,office,office
9,aajtak,news,rest


In [83]:
df_explode = pd.merge(df_appography_explode,
                      category_list,
                      how='left',
                      left_on='app_list_set',
                      right_on='app_name'
                     )
df_explode.head(5)

,unlocked_customer_id,scratched_customer_id,rewards_customer_id,cta_clicked_user_id,copy_code_clicked_user_id,customer_id,app_list_set,app_name,categories_l1,categories_l2
0,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,google maps,google maps,tools,rest
1,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,spotify,spotify,streaming_music,rest
2,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,namma yatri,namma yatri,travel_ridehailing,ride haling
3,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,bookmyshow,bookmyshow,entertainment,rest
4,5bc74480265bc271049bf01a,5bc74480265bc271049bf01a,NaN,NaN,NaN,5bc74480265bc271049bf01a,messenger,messenger,social,rest


In [84]:
df_summary = df_explode\
                .groupby(['categories_l1'])\
                .agg(scratched_customers = ('scratched_customer_id', 'nunique'),
#                      rewards_customers = ('rewards_customer_id', 'nunique'),
                     cta_customers = ('cta_clicked_user_id', 'nunique'),
                     copy_code_clicked_customers = ('copy_code_clicked_user_id', 'nunique'),
                    )\
                .reset_index()

In [85]:
df_summary

,categories_l1,scratched_customers,cta_customers,copy_code_clicked_customers
0,dating,12291,759,987
1,delivery_food,138048,9521,13301
2,delivery_grocery,79336,4893,6795
3,devotional,4,0,2
4,driver_app,15767,2038,2740
5,ecommerce,153239,12657,17639
6,educational,42836,2489,3599
7,entertainment,65123,4019,5507
8,fantasy_sports,6586,1593,2125
9,finance_investment,79163,4687,6551


In [86]:
# df_summary['% rewards pv'] = calculate_percentage(df_summary, 'rewards_customers', 'scratched_customers')
df_summary['% cta'] = calculate_percentage(df_summary, 'cta_customers', 'scratched_customers')
df_summary['% copy_code_clicked'] = calculate_percentage(df_summary, 'copy_code_clicked_customers', 'scratched_customers')


In [106]:
df_summary.rename(columns={'categories_l1': 'category/app'}, inplace=True)
df_summary.sort_values(by='% copy_code_clicked', ascending=False)

,category/app,scratched_customers,cta_customers,copy_code_clicked_customers,% cta,% copy_code_clicked
3,devotional,4,0,2,0.00,50.00
8,fantasy_sports,6586,1593,2125,24.19,32.27
4,driver_app,15767,2038,2740,12.93,17.38
12,gaming,25662,2491,3408,9.71,13.28
14,news,64191,5549,7777,8.64,12.12
17,social,189476,16300,22908,8.60,12.09
19,tools,189758,16331,22945,8.61,12.09
5,ecommerce,153239,12657,17639,8.26,11.51
16,ott,158148,12906,18121,8.16,11.46
18,streaming_music,131230,10426,14632,7.94,11.15


In [91]:
df_app_summary = df_explode[df_explode['app_list_set'].isin(['my11circle'])]\
                .groupby(['app_list_set'])\
                .agg(scratched_customers = ('scratched_customer_id', 'nunique'),
#                      rewards_customers = ('rewards_customer_id', 'nunique'),
                     cta_customers = ('cta_clicked_user_id', 'nunique'),
                     copy_code_clicked_customers = ('copy_code_clicked_user_id', 'nunique'),
                    )\
                .reset_index()

In [92]:
df_app_summary

,app_list_set,scratched_customers,cta_customers,copy_code_clicked_customers
0,my11circle,6224,1563,2084


In [93]:
# df_summary['% rewards pv'] = calculate_percentage(df_summary, 'rewards_customers', 'scratched_customers')
df_app_summary['% cta'] = calculate_percentage(df_app_summary, 'cta_customers', 'scratched_customers')
df_app_summary['% copy_code_clicked'] = calculate_percentage(df_app_summary, 'copy_code_clicked_customers', 'scratched_customers')

In [107]:
df_app_summary.rename(columns={'app_list_set': 'category/app'}, inplace=True)
df_app_summary.sort_values(by='% copy_code_clicked', ascending=False)


,category/app,scratched_customers,cta_customers,copy_code_clicked_customers,% cta,% copy_code_clicked
0,my11circle,6224,1563,2084,25.11,33.48


In [108]:
concatenated_df = pd.concat([df_summary, df_app_summary], ignore_index=False)

In [110]:
concatenated_df.sort_values(by='% copy_code_clicked', ascending=False)

,category/app,scratched_customers,cta_customers,copy_code_clicked_customers,% cta,% copy_code_clicked
3,devotional,4,0,2,0.00,50.00
0,my11circle,6224,1563,2084,25.11,33.48
8,fantasy_sports,6586,1593,2125,24.19,32.27
4,driver_app,15767,2038,2740,12.93,17.38
12,gaming,25662,2491,3408,9.71,13.28
14,news,64191,5549,7777,8.64,12.12
17,social,189476,16300,22908,8.60,12.09
19,tools,189758,16331,22945,8.61,12.09
5,ecommerce,153239,12657,17639,8.26,11.51
16,ott,158148,12906,18121,8.16,11.46


# Previous analysis

In [4]:
df_data['app_list'] = df_data['app_list'].str.strip()
df_data.drop_duplicates(inplace=True)

df_merge = pd.merge(df_data, 
                    category_list,
                    how='left',
                    left_on='app_list',
                    right_on='app_name'
                   )
df_merge['values'] = 1
df_merge

,user_id,cta_clicked,copy_code_clicked,id,app_list,app_name,categories_l1,categories_l2,values
0,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,amazon prime video,amazon prime video,ott,rest,1
1,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
2,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
3,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,alt balaji,alt balaji,ott,rest,1
4,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,bookmyshow,bookmyshow,entertainment,rest,1
...,...,...,...,...,...,...,...,...,...
2331347,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola,ola,travel_ridehailing,ride haling,1
2331348,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola driver app,ola driver app,driver_app,driver_app,1
2331349,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,rapido captain,rapido captain,driver_app,driver_app,1
2331350,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,snapchat,snapchat,social,rest,1


### Apps

In [5]:
df1 = df_merge.groupby(['app_list'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df1['total_customers'] = df_merge.user_id.nunique()
df1['%'] = (df1['count']*100.00/df1['total_customers']).round(2) 

In [7]:
df1.sort_values(by='%', ascending=False).head(60)

,app_list,count,total_customers,%
0,google maps,100722,100770,99.95
1,youtube,100474,100770,99.71
2,gmail,100429,100770,99.66
3,whatsapp,97229,100770,96.49
4,google photos,93904,100770,93.19
5,instagram,81833,100770,81.21
6,facebook,73284,100770,72.72
7,truecaller,67906,100770,67.39
8,google calendar,61272,100770,60.80
9,flipkart,58242,100770,57.80


### Categories

In [9]:
df2 = df_merge.groupby(['categories_l1'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df2['total_customers'] = df_merge.user_id.nunique()
df2['%'] = (df2['count']*100.00/df2['total_customers']).round(2) 

In [10]:
df2.sort_values(by='%', ascending=False).head(60)

,categories_l1,count,total_customers,%
0,tools,100765,100770,100.00
1,social,100592,100770,99.82
2,ott,78234,100770,77.64
3,ecommerce,72496,100770,71.94
4,finance_transactions,64575,100770,64.08
5,travel_ridehailing,61721,100770,61.25
6,streaming_music,60862,100770,60.40
7,delivery_food,54651,100770,54.23
8,office,45440,100770,45.09
9,news,31105,100770,30.87


### Remove most common apps

In [53]:
most_common_apps = ['google maps', 'youtube', 'gmail', 'whatsapp', 'google photos', 
                    'google calendar','clock', 'google news', 'hangouts', 'calculator',
                    'accuweather', 'truecaller'
                   ]

In [35]:
df3 = df_merge[~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['app_list'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df3['total_customers'] = df_merge.user_id.nunique()
df3['%'] = (df3['count']*100.00/df3['total_customers']).round(2) 

In [36]:
df3.head(40)

,app_list,count,total_customers,%
0,instagram,81833,100770,81.21
1,facebook,73284,100770,72.72
2,truecaller,67906,100770,67.39
3,flipkart,58242,100770,57.80
4,paytm money,58140,100770,57.70
5,jio cinema,52198,100770,51.80
6,snapchat,51236,100770,50.84
7,telegram,48416,100770,48.05
8,disney+ hotstar,46718,100770,46.36
9,spotify,46294,100770,45.94


In [37]:
df4 = df_merge[~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['categories_l1'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df4['total_customers'] = df_merge.user_id.nunique()
df4['%'] = (df4['count']*100.00/df4['total_customers']).round(2) 

In [38]:
df4.sort_values(by='%', ascending=False).head(60)

,categories_l1,count,total_customers,%
0,social,98018,100770,97.27
1,tools,79281,100770,78.68
2,ott,78234,100770,77.64
3,ecommerce,72496,100770,71.94
4,finance_transactions,64575,100770,64.08
5,travel_ridehailing,61721,100770,61.25
6,streaming_music,60862,100770,60.40
7,delivery_food,54651,100770,54.23
8,office,45440,100770,45.09
9,finance_investment,28357,100770,28.14


### Check

In [39]:
df5 = df_merge[~df_merge['copy_code_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['app_list'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df5['total_customers'] = df_merge.user_id.nunique()
df5['%'] = (df5['count']*100.00/df5['total_customers']).round(2) 

In [40]:
df5.head(40)

,app_list,count,total_customers,%
0,instagram,81763,100770,81.14
1,facebook,73220,100770,72.66
2,truecaller,67852,100770,67.33
3,flipkart,58194,100770,57.75
4,paytm money,58091,100770,57.65
5,jio cinema,52160,100770,51.76
6,snapchat,51190,100770,50.80
7,telegram,48381,100770,48.01
8,disney+ hotstar,46677,100770,46.32
9,spotify,46253,100770,45.90


In [41]:
df_merge[~df_merge['copy_code_clicked'].isna()]

,user_id,cta_clicked,copy_code_clicked,id,app_list,app_name,categories_l1,categories_l2,values
0,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,amazon prime video,amazon prime video,ott,rest,1
1,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
2,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
3,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,alt balaji,alt balaji,ott,rest,1
4,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,bookmyshow,bookmyshow,entertainment,rest,1
...,...,...,...,...,...,...,...,...,...
2331347,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola,ola,travel_ridehailing,ride haling,1
2331348,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola driver app,ola driver app,driver_app,driver_app,1
2331349,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,rapido captain,rapido captain,driver_app,driver_app,1
2331350,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,snapchat,snapchat,social,rest,1


In [42]:
df6 = df_merge[~df_merge['copy_code_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['categories_l1'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df6['total_customers'] = df_merge.user_id.nunique()
df6['%'] = (df6['count']*100.00/df6['total_customers']).round(2) 

In [43]:
df6.head(40)

,categories_l1,count,total_customers,%
0,social,97927,100770,97.18
1,tools,79217,100770,78.61
2,ott,78159,100770,77.56
3,ecommerce,72441,100770,71.89
4,finance_transactions,64521,100770,64.03
5,travel_ridehailing,61669,100770,61.20
6,streaming_music,60798,100770,60.33
7,delivery_food,54596,100770,54.18
8,office,45386,100770,45.04
9,finance_investment,28333,100770,28.12


In [44]:
category_list[category_list['categories_l1'] == 'finance_transactions']

,app_name,categories_l1,categories_l2
19,airtel thanks,finance_transactions,rest
42,axis mobile,finance_transactions,rest
84,cred,finance_transactions,rest
110,freecharge,finance_transactions,rest
131,bhim upi,finance_transactions,rest
132,google pay,finance_transactions,rest
201,mobikwik,finance_transactions,rest
239,paytm,finance_transactions,rest
241,paytm money,finance_transactions,rest
245,phonepe,finance_transactions,rest


In [47]:
# df_merge

In [58]:
df7 = df_merge[~df_merge['cta_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['categories_l1'])\
                .agg(apps  = ('app_name', set),
                     count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()
    
df7

,categories_l1,apps,count
0,social,"{linkedin, viber, mx takatak, sharechat, line,...",64093
1,ott,"{alt balaji, voot, amazon prime video, sonyliv...",51417
2,ecommerce,"{myntra, firstcry, meesho, nykaa, zivame, phar...",47854
3,finance_transactions,"{airtel thanks, freecharge, paytm money, mobik...",43547
4,travel_ridehailing,"{quick ride, in drive, blusmart, quickride, ub...",40441
5,streaming_music,"{wynk music, amazon music, spotify, hungama mu...",39672
6,delivery_food,"{swiggy, uber eats, zomato, dunzo}",35700
7,office,"{trello, zoho meeting, ms authenticator, onedr...",28726
8,tools,"{adobe scan, notion, file commander, google sh...",26799
9,finance_investment,"{kite by zerodha, nps, angel broking, namma pa...",18955


In [59]:
df7['total_customers'] = df_merge[~df_merge['cta_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)].user_id.nunique()
df7['%'] = (df7['count']*100.00/df7['total_customers']).round(2) 

In [60]:
df7

,categories_l1,apps,count,total_customers,%
0,social,"{linkedin, viber, mx takatak, sharechat, line,...",64093,65603,97.70
1,ott,"{alt balaji, voot, amazon prime video, sonyliv...",51417,65603,78.38
2,ecommerce,"{myntra, firstcry, meesho, nykaa, zivame, phar...",47854,65603,72.94
3,finance_transactions,"{airtel thanks, freecharge, paytm money, mobik...",43547,65603,66.38
4,travel_ridehailing,"{quick ride, in drive, blusmart, quickride, ub...",40441,65603,61.65
5,streaming_music,"{wynk music, amazon music, spotify, hungama mu...",39672,65603,60.47
6,delivery_food,"{swiggy, uber eats, zomato, dunzo}",35700,65603,54.42
7,office,"{trello, zoho meeting, ms authenticator, onedr...",28726,65603,43.79
8,tools,"{adobe scan, notion, file commander, google sh...",26799,65603,40.85
9,finance_investment,"{kite by zerodha, nps, angel broking, namma pa...",18955,65603,28.89
